# CoSense EN-NN — chuẩn bị dữ liệu và fine-tune B1/B2 trên Kaggle

**Đọc hướng dẫn `docs/KAGGLE_EN_NN_GUIDE.md` trước lần chạy đầu.**

Notebook tự chứa code; không cần clone GitHub, GitHub token hay API key.
Dùng Dataset tạo từ `dist/kaggle/cosense-en-nn-data.zip` và giữ notebook/Dataset private.
Chỉ upload sau khi kiểm tra điều khoản sử dụng dữ liệu của BTC.

Ba chế độ: `prepare` (mặc định, kiểm tra + tokenize + B0, không optimizer),
`smoke` (tối đa 128 mẫu train, vài bước optimizer/GPU), `train` (fine-tune đầy đủ).
Mọi đánh giá/chọn checkpoint chỉ dùng **validation**. Final chỉ được kiểm tra tính toàn vẹn;
không xuất dữ liệu train-ready, prediction hoặc metric final.

Lần đầu: import notebook → Add Input Dataset → Internet On → giữ `MODE="prepare"` → Run All.
Không cần GPU cho `prepare`. Chọn GPU trước `smoke`/`train` và chạy lại từ đầu sau khi session restart.


## 1. Cấu hình — sửa tại cell này

Lần đầu giữ nguyên. `DATA_ROOT=""` sẽ tự tìm manifest dù Kaggle đổi đường dẫn mount.
Nếu gắn nhiều bản Dataset, điền đường dẫn thư mục chứa `bundle_manifest.json`.
`unseen_compound` là split B chính; `seen_context` là split A bổ sung.
Một GPU được dùng, kể cả khi Kaggle cấp T4 x2. Chưa chạy distributed/multi-GPU.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import sys

MODE = "prepare"                    # "prepare", "smoke", "train"
SETTING = "unseen_compound"          # hoặc "seen_context"
VARIANTS = ["B1", "B2"]
SEED = 42
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
MAX_LENGTH = 256
EPOCHS = 5
PATIENCE = 2
SMOKE_STEPS = 8
MODEL_ID = "distilbert/distilbert-base-uncased"
MODEL_REVISION = "main"              # lần đầu resolve SHA; pin SHA đó cho các lần sau
INSTALL_DEPS = True
DATA_ROOT = ""                      # để trống để tự tìm; không dùng đường dẫn Windows ở Kaggle
RESUME_FROM = {"B1": "", "B2": ""}  # đường dẫn .../B1/last.pt; giữ cả best.safetensors cạnh nó
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")

if MODE not in {"prepare", "smoke", "train"}:
    raise ValueError("MODE không hợp lệ")
if SETTING not in {"unseen_compound", "seen_context"}:
    raise ValueError("SETTING không hợp lệ")
if not VARIANTS or len(set(VARIANTS)) != len(VARIANTS) or not set(VARIANTS) <= {"B1", "B2"}:
    raise ValueError("VARIANTS chỉ nhận B1/B2, không lặp")
if min(BATCH_SIZE, MAX_LENGTH, EPOCHS, PATIENCE, SMOKE_STEPS) < 1 or LEARNING_RATE <= 0:
    raise ValueError("Cấu hình số phải dương")
if not INPUT.is_dir():
    raise RuntimeError("Chạy notebook trên Kaggle. Local chỉ build/verify; không train trên CPU local.")
WORK.mkdir(parents=True, exist_ok=True)
OUTPUT = WORK / "cosense_outputs" / f"{SETTING}_{MODE}_seed{SEED}_{RUN_TAG}"
OUTPUT.mkdir(parents=True, exist_ok=False)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
print("Mode:", MODE, "| setting:", SETTING, "| output:", OUTPUT)


## 2. Cài dependency và kiểm tra runtime

Giữ PyTorch CUDA có sẵn trên Kaggle. Cell chỉ cài ba dependency đã pin.
Đây là cấu hình khởi đầu, không khẳng định tương thích mọi Kaggle image.
Nếu đã import transformers/torch trước khi đổi dependency, restart session rồi Run All.
Nếu Internet bị tắt thì pip/tải tokenizer lần đầu sẽ thất bại.


In [ ]:
import subprocess
REQUIREMENTS = "# Keep Kaggle's preinstalled CUDA-enabled torch. Do not pip-install CPU torch.\ntransformers==4.57.1\nhuggingface-hub==0.35.3\nsafetensors==0.6.2\n"
requirements_path = WORK / "requirements-cosense.txt"
requirements_path.write_text(REQUIREMENTS, encoding="utf-8")
if INSTALL_DEPS:
    if any(name in sys.modules for name in ("transformers", "tokenizers", "huggingface_hub")):
        raise RuntimeError("Dependency đã import. Restart session trước khi cài/đổi phiên bản.")
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                    "-r", str(requirements_path)], check=True)
import torch
print("Python:", sys.version.split()[0], "| torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU 0:", torch.cuda.get_device_name(0))
if MODE != "prepare" and not torch.cuda.is_available():
    raise RuntimeError("Bật Accelerator GPU rồi restart session trước smoke/train.")


## 3. Nạp code đi kèm notebook — không cần sửa cell này

Nguồn được nhúng từ `scripts/data_utils.py` và `scripts/kaggle_en_nn.py` lúc build.
Notebook ghi code vào thư mục working riêng, kiểm tra hash rồi import.
Muốn cập nhật code: sửa source local, build lại notebook, import lại bản mới.


In [ ]:
import hashlib
import importlib
EMBEDDED_SOURCES = {'data_utils.py': '"""Shared, standard-library-only audit/split/evaluation primitives."""\nimport csv\nimport hashlib\nimport json\nimport math\nimport random\nimport re\nimport statistics as st\nimport unicodedata\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\nNN = [\'ContextID\',\'Compound\',\'Mod\',\'Head\',\'ModAvg\',\'ModStd\',\'HeadAvg\',\'HeadStd\',\'Context\']\nPV = [\'ContextID\',\'ParticleVerb\',\'Base\',\'Particle\',\'Avg\',\'Std\',\'Context\']\nSPLITS = (\'train\',\'validation\',\'final\')\n\ndef config(path=None):\n    return json.loads((Path(path) if path else ROOT/\'configs/data_protocol.json\').read_text(encoding=\'utf-8\'))\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef norm(text):\n    return \' \'.join(unicodedata.normalize(\'NFC\', text).casefold().split())\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\ndef write_json(path, value):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    path.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False)+\'\\n\',encoding=\'utf-8\')\n\ndef write_csv(path, rows, fields, delimiter=\',\'):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    with path.open(\'w\',encoding=\'utf-8\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=fields,delimiter=delimiter,extrasaction=\'ignore\')\n        w.writeheader(); w.writerows(rows)\n\ndef load(branch, cfg=None):\n    cfg=cfg or config(); path=ROOT/cfg[\'sources\'][branch]\n    expected=NN if branch==\'en_nn\' else PV\n    issues=[]; rows=[]\n    with path.open(encoding=cfg[\'parser\'][\'encoding\'],newline=\'\') as f:\n        reader=csv.DictReader(f,delimiter=\'\\t\',quotechar=\'"\',strict=True)\n        fields=reader.fieldnames\n        if fields!=expected: issues.append({\'type\':\'schema\',\'expected\':expected,\'actual\':fields})\n        try:\n            for index,r in enumerate(reader,1):\n                if None in r or any(v is None for v in r.values()):\n                    issues.append({\'record\':index,\'type\':\'width\'}); continue\n                r[\'record_index\']=index; r[\'source_file\']=cfg[\'sources\'][branch]\n                r[\'row_key\']=f"{branch}:{r.get(\'ContextID\',\'\')}"\n                rows.append(r)\n        except csv.Error as e:\n            issues.append({\'type\':\'csv_parse\',\'line\':reader.line_num,\'message\':str(e)})\n    return rows,fields,issues\n\ndef labels(branch):\n    return [\'ModAvg\',\'HeadAvg\'] if branch==\'en_nn\' else [\'Avg\']\n\ndef target(branch):\n    return \'Compound\' if branch==\'en_nn\' else \'ParticleVerb\'\n\ndef quantile(vals,p):\n    if not vals:return None\n    vals=sorted(vals); pos=(len(vals)-1)*p; lo=int(pos); hi=min(lo+1,len(vals)-1)\n    return vals[lo]+(vals[hi]-vals[lo])*(pos-lo)\n\ndef stats(vals):\n    if not vals:return {\'n\':0}\n    return {\'n\':len(vals),\'min\':min(vals),\'max\':max(vals),\'mean\':st.mean(vals),\'median\':st.median(vals),\n            \'std_population\':st.pstdev(vals),\'quantiles\':{str(p):quantile(vals,p) for p in [0,.05,.25,.5,.75,.95,1]}}\n\ndef correlation(x,y):\n    if len(x)<2 or st.pstdev(x)==0 or st.pstdev(y)==0:return None\n    return st.correlation(x,y)\n\ndef ranks(vals):\n    order=sorted(range(len(vals)),key=lambda i:vals[i]); out=[0.0]*len(vals); start=0\n    while start<len(order):\n        end=start+1\n        while end<len(order) and vals[order[end]]==vals[order[start]]:end+=1\n        for i in order[start:end]:out[i]=(start+end-1)/2+1\n        start=end\n    return out\n\nclass UnionFind:\n    def __init__(self,n):self.parent=list(range(n))\n    def find(self,i):\n        while self.parent[i]!=i:\n            self.parent[i]=self.parent[self.parent[i]]; i=self.parent[i]\n        return i\n    def union(self,a,b):\n        a,b=self.find(a),self.find(b)\n        if a!=b:self.parent[max(a,b)]=min(a,b)\n    def groups(self):\n        groups=defaultdict(list)\n        for i in range(len(self.parent)):groups[self.find(i)].append(i)\n        return sorted(groups.values(),key=lambda g:g[0])\n\ndef duplicate_groups(rows,branch,fields):\n    t=target(branch)\n    funcs={\'id\':lambda r:r[\'ContextID\'], \'full_row\':lambda r:tuple(r[c] for c in fields),\n           \'target_context_raw\':lambda r:(r[t],r[\'Context\']),\n           \'context_raw\':lambda r:r[\'Context\'], \'context_normalized\':lambda r:norm(r[\'Context\']),\n           \'target_context_normalized\':lambda r:(norm(r[t]),norm(r[\'Context\']))}\n    out={}\n    for kind,fn in funcs.items():\n        buckets=defaultdict(list)\n        for i,r in enumerate(rows):buckets[fn(r)].append(i)\n        out[kind]=[g for g in buckets.values() if len(g)>1]\n    conflicts=[]\n    for g in out[\'target_context_normalized\']:\n        if len({tuple(r[c] for c in labels(branch)) for r in [rows[i] for i in g]})>1:conflicts.append(g)\n    out[\'conflicting_labels\']=conflicts\n    out[\'context_different_target\']=[g for g in out[\'context_normalized\'] if len({norm(rows[i][t]) for i in g})>1]\n    return out\n\ndef near_candidates(rows,cfg):\n    """Two independent inverted indexes: complete at configured lexical thresholds.\n\n    This does not detect all paraphrases or semantic equivalence.\n    """\n    texts=[norm(r[\'Context\']) for r in rows]\n    tokens=[set(re.findall(r\'\\w+\',s)) for s in texts]\n    grams=[{s[i:i+5] for i in range(max(1,len(s)-4))} for s in texts]\n    inverted=defaultdict(list)\n    for i,ts in enumerate(tokens):\n        for t in ts:inverted[t].append(i)\n    settings=cfg[\'near_duplicate\']; shortlisted=set()\n    for i,ts in enumerate(tokens):\n        hits=Counter(j for t in ts for j in inverted[t] if j>i)\n        for j,n in sorted(hits.items()):\n            if texts[i]==texts[j]:continue\n            tj=n/(len(ts)+len(tokens[j])-n)\n            if tj>=settings[\'token_jaccard\']:shortlisted.add((i,j))\n    inverted=defaultdict(list)\n    for i,gg in enumerate(grams):\n        for g in gg:inverted[g].append(i)\n    tested=0\n    for i,gg in enumerate(grams):\n        hits=Counter(j for g in gg for j in inverted[g] if j>i)\n        for j,n in hits.items():\n            if texts[i]==texts[j]:continue\n            tested+=1\n            cj=n/(len(gg)+len(grams[j])-n)\n            if cj>=settings[\'char_5gram_jaccard\']:shortlisted.add((i,j))\n    out=[]\n    for i,j in sorted(shortlisted):\n        tj=len(tokens[i]&tokens[j])/len(tokens[i]|tokens[j]); cj=len(grams[i]&grams[j])/len(grams[i]|grams[j])\n        out.append({\'left\':i,\'right\':j,\'left_id\':rows[i][\'ContextID\'],\'right_id\':rows[j][\'ContextID\'],\n                    \'token_jaccard\':tj,\'char_5gram_jaccard\':cj,\'left_context\':rows[i][\'Context\'],\n                    \'right_context\':rows[j][\'Context\'],\'review_status\':\'unconfirmed_semantics_conservatively_grouped\'})\n    return out,tested\n\n# Deliberately small and transparent heuristic lexicon, not an NLP lemmatizer.\nIRREGULAR=dict(pair.split(\':\') for pair in (\'be:am,is,are,was,were,been,being have:has,had do:does,did,done go:goes,went,gone \'\n \'take:took,taken come:came,come get:got,gotten put:put set:set cut:cut let:let hit:hit shut:shut hurt:hurt \'\n \'run:ran,run sit:sat stand:stood give:gave,given make:made find:found leave:left keep:kept \'\n \'bring:brought buy:bought think:thought catch:caught teach:taught fight:fought seek:sought \'\n \'tell:told sell:sold say:said pay:paid lay:laid lie:lay,lain hold:held read:read lead:led \'\n \'write:wrote,written drive:drove,driven ride:rode,ridden rise:rose,risen fall:fell,fallen \'\n \'break:broke,broken speak:spoke,spoken throw:threw,thrown blow:blew,blown grow:grew,grown \'\n \'know:knew,known draw:drew,drawn eat:ate,eaten beat:beat,beaten bite:bit,bitten hide:hid,hidden \'\n \'shake:shook,shaken steal:stole,stolen wear:wore,worn tear:tore,torn strike:struck,stricken \'\n \'hang:hung choose:chose,chosen freeze:froze,frozen forget:forgot,forgotten lose:lost \'\n \'send:sent spend:spent build:built feel:felt meet:met sleep:slept sweep:swept win:won \'\n \'swim:swam,swum sing:sang,sung ring:rang,rung drink:drank,drunk sink:sank,sunk\').split())\n\ndef forms(word,verb=False):\n    w=norm(word); out={w,w+\'s\'}\n    if w.endswith((\'s\',\'x\',\'z\',\'ch\',\'sh\')):out.add(w+\'es\')\n    if w.endswith(\'y\') and len(w)>1 and w[-2] not in \'aeiou\':out.add(w[:-1]+\'ies\')\n    if verb:\n        out.update(IRREGULAR.get(w,\'\').split(\',\')); out.update([w+\'ed\',w+\'ing\'])\n        if w.endswith(\'e\'):out.update([w+\'d\',w[:-1]+\'ing\'])\n        if w.endswith(\'y\'):out.add(w[:-1]+\'ied\')\n        if len(w)>2 and w[-1] not in \'aeiouwxy\' and w[-2] in \'aeiou\' and w[-3] not in \'aeiou\':\n            out.update([w+w[-1]+\'ed\',w+w[-1]+\'ing\'])\n    else:\n        out.update({\'man\':\'men\',\'woman\':\'women\',\'child\':\'children\',\'foot\':\'feet\',\'tooth\':\'teeth\',\'mouse\':\'mice\'}.get(w,\'\').split(\',\'))\n    return out-{\'\'}\n\ndef alignment(r,branch,cfg):\n    phrase=r[target(branch)]; context=r[\'Context\']; pat=r\'(?<!\\w)\'+re.escape(phrase)+r\'(?!\\w)\'\n    exact=list(re.finditer(pat,context)); insensitive=list(re.finditer(pat,context,re.IGNORECASE))\n    matches=insensitive\n    if matches:\n        return {\'match_class\':\'multiple_candidates\' if len(matches)>1 else (\'exact_contiguous\' if exact else \'case_insensitive_contiguous\'),\n                \'evidence\':\'surface_with_word_boundaries\',\'candidates\':[[[m.start(),m.end()]] for m in matches],\n                \'selected_span\':[[matches[0].start(),matches[0].end()]] if len(matches)==1 else None}\n    words=list(re.finditer(r"\\w+(?:[\'’]\\w+)?",context)); values=[norm(m.group()) for m in words]\n    left=r[\'Mod\' if branch==\'en_nn\' else \'Base\']; right=r[\'Head\' if branch==\'en_nn\' else \'Particle\']\n    if len(re.findall(r\'\\w+\',left))!=1 or len(re.findall(r\'\\w+\',right))!=1:\n        return {\'match_class\':\'unresolved\',\'evidence\':\'multiword_component_not_supported\',\'candidates\':[],\'selected_span\':None}\n    lf=forms(left,branch==\'en_pv\'); rf=forms(right,False) if branch==\'en_nn\' else {norm(right)}\n    candidates=[]; adjacent=[]\n    for i,w in enumerate(values):\n        if w not in lf:continue\n        for j,v in enumerate(values):\n            if j<=i or v not in rf:continue\n            distance=j-i\n            if distance==1 or (branch==\'en_pv\' and distance<=cfg[\'alignment\'][\'pv_max_token_distance\']):\n                span=[[words[i].start(),words[i].end()],[words[j].start(),words[j].end()]]\n                candidates.append(span)\n                if distance==1:adjacent.append(span)\n    # Multiple alternatives are retained, never silently choose the first.\n    if len(candidates)>1:kind=\'multiple_candidates\'\n    elif adjacent:kind=\'inflection_or_punctuation_candidate\'\n    elif candidates:kind=\'discontinuous_candidate\'\n    else:kind=\'unresolved\'\n    return {\'match_class\':kind,\'evidence\':\'heuristic_not_linguistic_confirmation\',\'candidates\':candidates,\'selected_span\':None}\n\ndef build_groups(rows,near,compound=False):\n    uf=UnionFind(len(rows)); buckets=defaultdict(list)\n    for i,r in enumerate(rows):buckets[norm(r[\'Context\'])].append(i)\n    for group in buckets.values():\n        for i in group[1:]:uf.union(group[0],i)\n    for pair in near:uf.union(pair[\'left\'],pair[\'right\'])\n    if compound:\n        buckets=defaultdict(list)\n        for i,r in enumerate(rows):buckets[norm(r[\'Compound\'])].append(i)\n        for group in buckets.values():\n            for i in group[1:]:uf.union(group[0],i)\n    return uf.groups()\n\ndef allocate(rows,near,cfg,setting):\n    groups=build_groups(rows,near,compound=setting==\'unseen_compound\')\n    rng=random.Random(cfg[\'split\'][\'seed\']); shuffled=groups.copy(); rng.shuffle(shuffled)\n    assignment={}; comp_groups=defaultdict(set)\n    for gi,g in enumerate(groups):\n        for i in g:comp_groups[norm(rows[i][\'Compound\'])].add(gi)\n    forced=set()\n    if setting==\'seen_context\':\n        for memberships in comp_groups.values():\n            if len(memberships)<=cfg[\'split\'][\'small_compound_groups_train_only\']:forced.update(memberships)\n        # Choose train coverage from seed-shuffled order without using labels.\n        covered=set()\n        for gi in sorted(forced):\n            assignment[gi]=\'train\'; covered.update(norm(rows[i][\'Compound\']) for i in groups[gi])\n        gindex={tuple(g):gi for gi,g in enumerate(groups)}\n        for g in shuffled:\n            gi=gindex[tuple(g)]; compounds={norm(rows[i][\'Compound\']) for i in g}\n            if not compounds<=covered:\n                assignment[gi]=\'train\'; covered.update(compounds)\n    counts=Counter()\n    for gi,s in assignment.items():counts[s]+=len(groups[gi])\n    # Seed-shuffled order avoids routing most large compounds into train.\n    # Balance record counts, never use labels or baseline/model scores.\n    ordered=shuffled; gindex={tuple(g):gi for gi,g in enumerate(groups)}\n    goals={s:len(rows)*cfg[\'split\'][\'ratios\'][s] for s in SPLITS}\n    for g in ordered:\n        gi=gindex[tuple(g)]\n        if gi in assignment:continue\n        s=min(SPLITS,key=lambda s:sum((counts[k]+(len(g) if k==s else 0)-goals[k])**2 for k in SPLITS))\n        assignment[gi]=s; counts[s]+=len(g)\n    members={s:[] for s in SPLITS}\n    for gi,g in enumerate(groups):\n        for i in g:\n            members[assignment[gi]].append({\'ContextID\':rows[i][\'ContextID\'],\'row_key\':rows[i][\'row_key\'],\n              \'Compound\':rows[i][\'Compound\'],\'Compound_normalized\':norm(rows[i][\'Compound\']),\n              \'group_id\':f\'g{gi:05d}\',\'source_file\':rows[i][\'source_file\']})\n    for s in SPLITS:members[s].sort(key=lambda r:r[\'ContextID\'])\n    return members,{\'groups\':len(groups),\'max_group_records\':max(map(len,groups)),\n                    \'forced_train_only_groups\':len(forced),\'algorithm\':\'seed42_shuffled_group_squared_row_deficit_v2\'}\n\ndef source_snapshot(cfg):\n    return {b:{\'path\':p,\'bytes\':(ROOT/p).stat().st_size,\'sha256\':digest(ROOT/p)} for b,p in cfg[\'sources\'].items()}\n\ndef split_dir(setting):return ROOT/\'data/splits/en_nn\'/setting\n\ndef read_members(setting,s):\n    with (split_dir(setting)/f\'{s}.csv\').open(encoding=\'utf-8\',newline=\'\') as f:return list(csv.DictReader(f))\n\ndef metric(rows,predictions):\n    result={\'n\':len(rows),\'compounds\':len({norm(r[\'Compound\']) for r in rows}),\'outputs\':{}}\n    row_maes=[]\n    for c,idx in [(\'ModAvg\',0),(\'HeadAvg\',1)]:\n        y=[float(r[c]) for r in rows]; p=[v[idx] for v in predictions]\n        errors=[abs(a-b) for a,b in zip(y,p)]\n        result[\'outputs\'][c]={\'mae\':st.mean(errors),\'rmse\':math.sqrt(st.mean((a-b)**2 for a,b in zip(y,p))),\n          \'pearson\':correlation(y,p),\'spearman\':correlation(ranks(y),ranks(p)),\n          \'correlation_status\':\'NA: constant predictions\' if st.pstdev(p)==0 else \'defined_or_constant_gold\'}\n        row_maes.append(errors)\n    per_row=[(a+b)/2 for a,b in zip(*row_maes)]; by=defaultdict(list)\n    for r,e in zip(rows,per_row):by[norm(r[\'Compound\'])].append(e)\n    result[\'row_macro_two_output_mae\']=st.mean(per_row)\n    result[\'compound_macro_two_output_mae\']=st.mean(st.mean(v) for v in by.values())\n    return result\n', 'kaggle_en_nn.py': '"""EN-NN Kaggle preparation and optional training. Heavy imports stay lazy.\n\nThe notebook embeds this file and data_utils.py, so no GitHub access is needed.\nOnly train/validation are exported or scored. Final membership is integrity-only.\n"""\nimport csv\nimport importlib.metadata\nimport json\nimport math\nimport platform\nimport random\nimport statistics\nimport time\nimport zipfile\nfrom collections import Counter\nfrom pathlib import Path, PurePosixPath\n\nfrom data_utils import NN, SPLITS, allocate, digest, metric, norm, write_csv, write_json\n\nSETTINGS = (\'unseen_compound\', \'seen_context\')\nSOURCE = \'data/train/en-nn-train.tsv\'\nEXPECTED_SOURCE_SHA256 = \'98ca6db405a5cdea6cebaaed323ef6e50e3ddd2926b908bbfded8badeee8d895\'\nBUNDLE_VERSION = \'cosense-en-nn-kaggle-v1\'\n\n\ndef payload_paths():\n    paths = [SOURCE, \'configs/data_protocol.json\', \'reports/audit/en_nn_near_candidates.json\']\n    for setting in SETTINGS:\n        paths.extend(f\'data/splits/en_nn/{setting}/{name}\'\n                     for name in (\'train.csv\', \'validation.csv\', \'final.csv\', \'manifest.json\'))\n    return paths\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef read_csv(path, delimiter=\',\'):\n    with Path(path).open(encoding=\'utf-8-sig\', newline=\'\') as handle:\n        reader = csv.DictReader(handle, delimiter=delimiter, strict=True)\n        rows = list(reader)\n        if any(None in row or any(v is None for v in row.values()) for row in rows):\n            raise ValueError(f\'Malformed CSV/TSV: {path}\')\n        return reader.fieldnames, rows\n\n\ndef verify_bundle(root):\n    root = Path(root)\n    manifest = read_json(root / \'bundle_manifest.json\')\n    if manifest[\'bundle_version\'] != BUNDLE_VERSION:\n        raise ValueError(\'Unsupported bundle version; rebuild notebook and dataset together.\')\n    if set(manifest[\'files\']) != set(payload_paths()):\n        raise ValueError(\'Bundle allowlist differs from expected EN-NN files.\')\n    for relative, expected in manifest[\'files\'].items():\n        path = root / relative\n        if not path.is_file() or digest(path) != expected[\'sha256\'] or path.stat().st_size != expected[\'bytes\']:\n            raise ValueError(f\'Missing/changed file: {relative}. Re-upload original ZIP without CSV conversion.\')\n    if digest(root / SOURCE) != EXPECTED_SOURCE_SHA256:\n        raise ValueError(\'Unexpected EN-NN source revision. Do not silently replace raw data.\')\n    return manifest\n\n\ndef discover_bundle(input_root=\'/kaggle/input\', work_root=\'/kaggle/working\', explicit_root=\'\'):\n    """Accept Kaggle-expanded ZIP or an attached ZIP; require one unambiguous bundle."""\n    if explicit_root:\n        root = Path(explicit_root)\n        verify_bundle(root)\n        return root\n    input_root = Path(input_root)\n    expanded = sorted({p.parent for p in input_root.rglob(\'bundle_manifest.json\')})\n    if len(expanded) == 1:\n        verify_bundle(expanded[0])\n        return expanded[0]\n    if len(expanded) > 1:\n        raise ValueError(\'Multiple bundles attached. Set DATA_ROOT to one manifest parent directory.\')\n    archives = sorted(input_root.rglob(\'cosense-en-nn-data*.zip\'))\n    if len(archives) != 1:\n        raise FileNotFoundError(\'Attach one cosense-en-nn-data ZIP/Dataset, or set DATA_ROOT. \'\n                                f\'Found {len(archives)} matching ZIPs under {input_root}.\')\n    archive = archives[0]\n    root = Path(work_root) / \'input_cache\' / digest(archive)[:16]\n    allowed = set(payload_paths()) | {\'bundle_manifest.json\'}\n    with zipfile.ZipFile(archive) as zipped:\n        names = [info.filename for info in zipped.infolist()]\n        if set(names) != allowed or len(names) != len(allowed):\n            raise ValueError(\'ZIP contains unexpected, duplicate, or missing files.\')\n        for info in zipped.infolist():\n            part = PurePosixPath(info.filename)\n            if part.is_absolute() or \'..\' in part.parts or \'\\\\\' in info.filename:\n                raise ValueError(\'Unsafe ZIP member.\')\n            if (info.external_attr >> 16) & 0o170000 == 0o120000:\n                raise ValueError(\'ZIP symlinks are not supported.\')\n        if sum(info.file_size for info in zipped.infolist()) > 50_000_000:\n            raise ValueError(\'Unexpectedly large EN-NN bundle.\')\n        root.mkdir(parents=True, exist_ok=True)\n        # Every filename is checked against the fixed relative allowlist first.\n        for name in names:\n            destination = root / name\n            destination.parent.mkdir(parents=True, exist_ok=True)\n            destination.write_bytes(zipped.read(name))\n    verify_bundle(root)\n    return root\n\n\ndef load_checked(root, setting):\n    if setting not in SETTINGS:\n        raise ValueError(f\'Unknown setting: {setting}\')\n    bundle = verify_bundle(root)\n    root = Path(root)\n    cfg = read_json(root / \'configs/data_protocol.json\')\n    if cfg[\'version\'] != \'cosense-data-v2\' or cfg[\'sources\'][\'en_nn\'] != SOURCE:\n        raise ValueError(\'Unexpected protocol/source path.\')\n    fields, rows = read_csv(root / SOURCE, \'\\t\')\n    if fields != NN or not rows:\n        raise ValueError(\'Unexpected or empty NN schema.\')\n    for row in rows:\n        if any(not value.strip() for value in row.values()):\n            raise ValueError(\'Empty source field.\')\n        row[\'row_key\'] = f"en_nn:{row[\'ContextID\']}"\n        row[\'source_file\'] = SOURCE\n        for key in (\'ModAvg\', \'HeadAvg\', \'ModStd\', \'HeadStd\'):\n            value = float(row[key])\n            if not math.isfinite(value) or value < 0 or (key.endswith(\'Avg\') and value > 5):\n                raise ValueError(f\'Invalid source score: {key}\')\n    lookup = {row[\'row_key\']: row for row in rows}\n    if len(lookup) != len(rows):\n        raise ValueError(\'Duplicate source ContextID.\')\n    directory = root / \'data/splits/en_nn\' / setting\n    split_manifest = read_json(directory / \'manifest.json\')\n    if (split_manifest[\'config\'] != cfg or split_manifest[\'setting\'] != setting\n            or split_manifest[\'sources\'][\'en_nn\'][\'sha256\'] != digest(root / SOURCE)):\n        raise ValueError(\'Stale split manifest/config/source.\')\n    members = {}\n    for split in SPLITS:\n        path = directory / f\'{split}.csv\'\n        if digest(path) != split_manifest[\'hashes\'][split]:\n            raise ValueError(f\'Changed membership: {split}\')\n        _, members[split] = read_csv(path)\n        if not members[split]:\n            raise ValueError(f\'Empty membership: {split}\')\n    keys = [member[\'row_key\'] for split in SPLITS for member in members[split]]\n    if len(keys) != len(set(keys)) or set(keys) != set(lookup):\n        raise ValueError(\'Split membership duplicate/missing/unrecognized row keys.\')\n    for split in SPLITS:\n        for member in members[split]:\n            row = lookup[member[\'row_key\']]\n            if (any(member[key] != row[key] for key in (\'ContextID\', \'Compound\', \'source_file\'))\n                    or member[\'Compound_normalized\'] != norm(row[\'Compound\'])):\n                raise ValueError(\'Membership source fields differ.\')\n    near = read_json(root / \'reports/audit/en_nn_near_candidates.json\')\n    replay, _ = allocate(rows, near, cfg, setting)\n    if replay != members:\n        raise ValueError(\'Deterministic split replay failed; do not re-split on Kaggle.\')\n    contexts = {s: {norm(lookup[m[\'row_key\']][\'Context\']) for m in members[s]} for s in SPLITS}\n    compounds = {s: {norm(lookup[m[\'row_key\']][\'Compound\']) for m in members[s]} for s in SPLITS}\n    for i, left in enumerate(SPLITS):\n        for right in SPLITS[i + 1:]:\n            if contexts[left] & contexts[right]:\n                raise ValueError(\'Context leakage between splits.\')\n            if setting == \'unseen_compound\' and compounds[left] & compounds[right]:\n                raise ValueError(\'Compound leakage in split B.\')\n    if setting == \'seen_context\' and any(not compounds[s] <= compounds[\'train\'] for s in (\'validation\', \'final\')):\n        raise ValueError(\'Split A must have train coverage for evaluation compounds.\')\n    # Final labels were checked for validity only. No final feature/target export.\n    selected = {s: [lookup[m[\'row_key\']] for m in members[s]] for s in (\'train\', \'validation\')}\n    report = {\'setting\': setting, \'source_sha256\': digest(root / SOURCE),\n              \'split_manifest_sha256\': digest(directory / \'manifest.json\'),\n              \'bundle_manifest_sha256\': digest(root / \'bundle_manifest.json\'),\n              \'bundle_version\': bundle[\'bundle_version\'], \'protocol\': cfg[\'version\'],\n              \'counts\': {s: len(members[s]) for s in SPLITS},\n              \'integrity_passed\': True, \'final_evaluated\': False,\n              \'final_access\': \'source validity and membership integrity only; no final export/prediction/metric\'}\n    return selected, report\n\n\ndef model_inputs(row, variant):\n    if variant == \'B1\':\n        return {\'text\': row[\'Compound\']}\n    if variant == \'B2\':\n        return {\'text\': row[\'Compound\'], \'text_pair\': row[\'Context\']}\n    raise ValueError(\'variant must be B1 or B2\')\n\n\ndef prepared_record(row, variant):\n    return {\'metadata\': {\'row_key\': row[\'row_key\'], \'ContextID\': row[\'ContextID\']},\n            \'inputs\': model_inputs(row, variant),\n            \'labels\': [float(row[\'ModAvg\']), float(row[\'HeadAvg\'])]}\n\n\ndef write_jsonl(path, records):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\'w\', encoding=\'utf-8\') as handle:\n        for record in records:\n            handle.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + \'\\n\')\n\n\ndef save_prepared(selected, report, output):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    write_json(output / \'integrity.json\', report)\n    for variant in (\'B1\', \'B2\'):\n        for split, rows in selected.items():\n            write_jsonl(output / \'prepared\' / variant / f\'{split}.jsonl\',\n                        (prepared_record(row, variant) for row in rows))\n    means = [statistics.mean(float(row[key]) for row in selected[\'train\'])\n             for key in (\'ModAvg\', \'HeadAvg\')]\n    result = {\'split\': \'validation\', \'train_means\': means,\n              \'metrics\': metric(selected[\'validation\'], [means[:] for _ in selected[\'validation\']]),\n              \'final_evaluated\': False}\n    write_json(output / \'b0_validation.json\', result)\n    return result\n\n\ndef tokenize_rows(rows, variant, tokenizer, max_length):\n    """Fail rather than silently crop an unreviewed target context."""\n    records, lengths = [], []\n    for row in rows:\n        encoding = tokenizer(**model_inputs(row, variant), padding=False, truncation=False,\n                             return_token_type_ids=False)\n        length = len(encoding[\'input_ids\'])\n        lengths.append(length)\n        if length > max_length:\n            raise ValueError(f"{row[\'row_key\']} {variant} has {length} tokens > {max_length}. "\n                             \'Review length settings for BOTH models; no automatic cropping.\')\n        records.append({\'input_ids\': encoding[\'input_ids\'], \'attention_mask\': encoding[\'attention_mask\'],\n                        \'labels\': [float(row[\'ModAvg\']), float(row[\'HeadAvg\'])]})\n    return records, {\'n\': len(lengths), \'min\': min(lengths), \'max\': max(lengths),\n                     \'mean\': statistics.mean(lengths), \'truncated\': 0}\n\n\ndef tokenize_all(selected, tokenizer, max_length, output):\n    data, report = {}, {}\n    for variant in (\'B1\', \'B2\'):\n        data[variant], report[variant] = {}, {}\n        for split, rows in selected.items():\n            records, stats = tokenize_rows(rows, variant, tokenizer, max_length)\n            data[variant][split], report[variant][split] = records, stats\n            write_jsonl(Path(output) / \'tokens\' / variant / f\'{split}.jsonl\', records)\n    write_json(Path(output) / \'token_lengths.json\', report)\n    return data, report\n\n\ndef environment_report():\n    result = {\'python\': platform.python_version(), \'platform\': platform.platform()}\n    for package in (\'torch\', \'transformers\', \'tokenizers\', \'huggingface-hub\', \'safetensors\'):\n        try:\n            result[package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            result[package] = None\n    return result\n\n\ndef regression_model(model_id=None, revision=None, config_dir=None):\n    import torch\n    from transformers import AutoConfig, AutoModel\n\n    class TwoScoreRegressor(torch.nn.Module):\n        def __init__(self):\n            super().__init__()\n            if config_dir:\n                self.encoder = AutoModel.from_config(AutoConfig.from_pretrained(config_dir))\n            else:\n                self.encoder = AutoModel.from_pretrained(model_id, revision=revision, use_safetensors=True)\n            self.head = torch.nn.Linear(self.encoder.config.hidden_size, 2)\n\n        def forward(self, input_ids, attention_mask):\n            states = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state\n            mask = attention_mask.unsqueeze(-1).to(states.dtype)\n            pooled = (states * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)\n            return self.head(pooled)\n\n    return TwoScoreRegressor()\n\n\ndef make_loader(records, tokenizer, batch_size, shuffle=False, generator=None):\n    import torch\n\n    def collate(batch):\n        features = [{key: row[key] for key in (\'input_ids\', \'attention_mask\')} for row in batch]\n        padded = tokenizer.pad(features, padding=True, return_tensors=\'pt\')\n        padded[\'labels\'] = torch.tensor([row[\'labels\'] for row in batch], dtype=torch.float32)\n        return padded\n\n    return torch.utils.data.DataLoader(records, batch_size=batch_size, shuffle=shuffle,\n                                       collate_fn=collate, num_workers=0, generator=generator)\n\n\ndef predict(model, records, tokenizer, batch_size, device):\n    import torch\n    model.eval()\n    predictions = []\n    with torch.inference_mode():\n        for batch in make_loader(records, tokenizer, batch_size):\n            inputs = {key: batch[key].to(device) for key in (\'input_ids\', \'attention_mask\')}\n            predictions.extend(model(**inputs).cpu().tolist())\n    return predictions\n\n\ndef score_predictions(rows, raw):\n    if len(raw) != len(rows) or any(len(pair) != 2 or not all(math.isfinite(v) for v in pair) for pair in raw):\n        raise ValueError(\'Invalid/missing/nonfinite predictions.\')\n    clipped = [[min(5.0, max(0.0, value)) for value in pair] for pair in raw]\n    scores = metric(rows, clipped)\n    scores[\'raw_metrics\'] = metric(rows, raw)\n    scores[\'out_of_range_fraction\'] = sum(value < 0 or value > 5 for pair in raw for value in pair) / (2 * len(raw))\n    return scores, clipped\n\n\ndef export_predictions(path, rows, raw):\n    scores, clipped = score_predictions(rows, raw)\n    records = []\n    for row, prediction, clip in zip(rows, raw, clipped):\n        records.append({\'row_key\': row[\'row_key\'], \'ContextID\': row[\'ContextID\'], \'Compound\': row[\'Compound\'],\n                        \'raw_ModAvg\': prediction[0], \'raw_HeadAvg\': prediction[1],\n                        \'pred_ModAvg\': clip[0], \'pred_HeadAvg\': clip[1],\n                        \'gold_ModAvg\': row[\'ModAvg\'], \'gold_HeadAvg\': row[\'HeadAvg\']})\n    write_csv(path, records, list(records[0]), delimiter=\'\\t\')\n    return scores\n\n\ndef train_variant(selected, tokens, tokenizer, cfg, output, integrity, resume_from=\'\'):\n    """Single-GPU FP32 pilot; checkpoints/resume at completed epoch boundaries."""\n    import torch\n    if cfg[\'mode\'] not in (\'smoke\', \'train\'):\n        raise ValueError(\'Training requires mode smoke or train.\')\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'No CUDA GPU. Choose a Kaggle GPU accelerator before training.\')\n    device = torch.device(\'cuda:0\')\n    seed = cfg[\'seed\']\n    random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.use_deterministic_algorithms(True, warn_only=True)\n    output = Path(output)\n    if output.exists():\n        raise FileExistsError(f\'Run directory exists: {output}. Use a new RUN_TAG; no silent overwrite.\')\n    output.mkdir(parents=True)\n    variant = cfg[\'variant\']\n    train_tokens = tokens[\'train\']\n    if cfg[\'mode\'] == \'smoke\':\n        train_tokens = train_tokens[:min(128, len(train_tokens))]\n    # Validation always remains the entire fixed validation membership.\n    model = regression_model(cfg[\'model_id\'], cfg[\'model_revision\']).to(device)\n    tokenizer.save_pretrained(output / \'tokenizer\')\n    model.encoder.config.save_pretrained(output / \'backbone_config\')\n    write_json(output / \'config.json\', cfg)\n    write_json(output / \'integrity.json\', integrity)\n    runtime = environment_report()\n    runtime.update({\'gpu\': torch.cuda.get_device_name(0), \'device_count\': torch.cuda.device_count(),\n                    \'used_gpu_count\': 1, \'precision\': \'float32\', \'torch_cuda\': torch.version.cuda,\n                    \'seed_reproducibility\': \'fixed RNG seeds; deterministic algorithms warn_only=True\'})\n    write_json(output / \'environment.json\', runtime)\n    write_json(output / \'status.json\', {\'status\': \'running\', \'final_evaluated\': False})\n    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg[\'learning_rate\'], weight_decay=0.01)\n    generator = torch.Generator().manual_seed(seed)\n    loader = make_loader(train_tokens, tokenizer, cfg[\'batch_size\'], shuffle=True, generator=generator)\n    epochs = 1 if cfg[\'mode\'] == \'smoke\' else cfg[\'epochs\']\n    steps_per_epoch = len(loader) if cfg[\'mode\'] == \'train\' else min(len(loader), cfg[\'smoke_steps\'])\n    total_steps = steps_per_epoch * epochs\n    warmup = max(1, int(total_steps * 0.1))\n\n    def lr_factor(step):\n        if step < warmup:\n            return (step + 1) / warmup\n        return max(0.0, (total_steps - step) / max(1, total_steps - warmup))\n\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)\n    start_epoch, best, stale, global_step = 0, math.inf, 0, 0\n    history = []\n    compatibility = {\'config\': cfg, \'integrity\': integrity}\n    if resume_from:\n        # Load ONLY checkpoints produced by this notebook and kept under your control.\n        checkpoint = torch.load(resume_from, map_location=\'cpu\', weights_only=False)\n        if checkpoint[\'compatibility\'] != compatibility:\n            raise ValueError(\'Resume config/source/split mismatch. Keep hyperparameters and mode unchanged.\')\n        if checkpoint[\'runtime_versions\'] != {key: runtime[key] for key in (\'torch\', \'transformers\')}:\n            raise ValueError(\'Resume torch/transformers version mismatch.\')\n        model.load_state_dict(checkpoint[\'model\'])\n        optimizer.load_state_dict(checkpoint[\'optimizer\'])\n        scheduler.load_state_dict(checkpoint[\'scheduler\'])\n        # Optimizer tensors loaded on CPU must follow their parameter device.\n        for state in optimizer.state.values():\n            for key, value in state.items():\n                if torch.is_tensor(value):\n                    # Non-capturable AdamW keeps its scalar step counter on CPU.\n                    state[key] = value.cpu() if key == \'step\' else value.to(device)\n        start_epoch = checkpoint[\'epoch\'] + 1\n        best, stale, global_step = checkpoint[\'best\'], checkpoint[\'stale\'], checkpoint[\'global_step\']\n        history = checkpoint[\'history\']\n        random.setstate(checkpoint[\'python_rng\'])\n        torch.set_rng_state(checkpoint[\'torch_rng\'])\n        torch.cuda.set_rng_state_all(checkpoint[\'cuda_rng\'])\n        generator.set_state(checkpoint[\'loader_rng\'])\n        source_dir = Path(resume_from).parent\n        best_file = source_dir / \'best.safetensors\'\n        if not best_file.is_file():\n            raise FileNotFoundError(\'Resume needs best.safetensors next to last.pt. Attach the complete run folder.\')\n        (output / \'best.safetensors\').write_bytes(best_file.read_bytes())\n        if start_epoch >= epochs or stale >= cfg[\'patience\']:\n            raise ValueError(\'Checkpoint already completed maximum epochs or early stopping; no resume needed.\')\n    from safetensors.torch import load_file, save_file\n    started = time.monotonic()\n    torch.cuda.reset_peak_memory_stats()\n    try:\n        for epoch in range(start_epoch, epochs):\n            model.train()\n            loss_sum, examples = 0.0, 0\n            for step, batch in enumerate(loader):\n                if step >= steps_per_epoch:\n                    break\n                inputs = {key: batch[key].to(device) for key in (\'input_ids\', \'attention_mask\')}\n                labels = batch[\'labels\'].to(device)\n                optimizer.zero_grad(set_to_none=True)\n                predictions = model(**inputs)\n                loss = torch.nn.functional.mse_loss(predictions, labels)\n                if not torch.isfinite(loss):\n                    raise FloatingPointError(\'Nonfinite loss; training stopped.\')\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                optimizer.step()\n                scheduler.step()\n                global_step += 1\n                loss_sum += loss.item() * len(labels)\n                examples += len(labels)\n            raw = predict(model, tokens[\'validation\'], tokenizer, cfg[\'batch_size\'], device)\n            scores, _ = score_predictions(selected[\'validation\'], raw)\n            value = scores[\'compound_macro_two_output_mae\']\n            improved = value < best\n            if improved:\n                best, stale = value, 0\n                save_file({key: value.detach().cpu().contiguous() for key, value in model.state_dict().items()},\n                          str(output / \'best.safetensors\'))\n            else:\n                stale += 1\n            history.append({\'epoch\': epoch + 1, \'train_mse\': loss_sum / examples,\n                            \'validation_E\': value, \'global_step\': global_step,\n                            \'elapsed_seconds_this_session\': time.monotonic() - started})\n            write_json(output / \'history.json\', history)\n            checkpoint = {\'model\': model.state_dict(), \'optimizer\': optimizer.state_dict(),\n                          \'scheduler\': scheduler.state_dict(), \'epoch\': epoch, \'global_step\': global_step,\n                          \'best\': best, \'stale\': stale, \'history\': history, \'compatibility\': compatibility,\n                          \'runtime_versions\': {key: runtime[key] for key in (\'torch\', \'transformers\')},\n                          \'python_rng\': random.getstate(), \'torch_rng\': torch.get_rng_state(),\n                          \'cuda_rng\': torch.cuda.get_rng_state_all(), \'loader_rng\': generator.get_state()}\n            temporary = output / \'last.tmp\'\n            torch.save(checkpoint, temporary)\n            temporary.replace(output / \'last.pt\')\n            print(variant, json.dumps(history[-1]), flush=True)\n            if stale >= cfg[\'patience\']:\n                break\n        model.load_state_dict(load_file(str(output / \'best.safetensors\'), device=\'cpu\'))\n        raw = predict(model, tokens[\'validation\'], tokenizer, cfg[\'batch_size\'], device)\n        scores = export_predictions(output / \'validation_predictions.tsv\', selected[\'validation\'], raw)\n        result = {\'variant\': variant, \'setting\': cfg[\'setting\'], \'seed\': seed, \'mode\': cfg[\'mode\'],\n                  \'split\': \'validation\', \'train_n_used\': len(train_tokens), \'metrics\': scores,\n                  \'best_validation_E\': best, \'final_evaluated\': False,\n                  \'elapsed_seconds_this_session\': time.monotonic() - started,\n                  \'peak_gpu_allocated_bytes\': torch.cuda.max_memory_allocated()}\n        write_json(output / \'metrics.json\', result)\n        write_json(output / \'status.json\', {\'status\': \'completed\', \'final_evaluated\': False})\n        return result\n    except BaseException as error:\n        write_json(output / \'status.json\', {\'status\': \'failed_or_interrupted\', \'error_type\': type(error).__name__,\n                                          \'resume\': \'last.pt resumes after the last fully completed epoch only\',\n                                          \'final_evaluated\': False})\n        raise\n    finally:\n        del model, optimizer\n        torch.cuda.empty_cache()\n\n\ndef load_for_inference(run_dir, device=\'cpu\'):\n    """Reload custom mean-pooling + two-output head, rather than AutoModel alone."""\n    from safetensors.torch import load_file\n    from transformers import AutoTokenizer\n    run_dir = Path(run_dir)\n    cfg = read_json(run_dir / \'config.json\')\n    model = regression_model(config_dir=run_dir / \'backbone_config\')\n    model.load_state_dict(load_file(str(run_dir / \'best.safetensors\'), device=\'cpu\'))\n    model.to(device).eval()\n    tokenizer = AutoTokenizer.from_pretrained(run_dir / \'tokenizer\', local_files_only=True)\n    return model, tokenizer, cfg\n\n\ndef predict_texts(run_dir, compounds, contexts=None, device=\'cpu\'):\n    model, tokenizer, cfg = load_for_inference(run_dir, device)\n    if not compounds or (cfg[\'variant\'] == \'B2\' and (contexts is None or len(contexts) != len(compounds))):\n        raise ValueError(\'Provide compounds; B2 needs exactly one context per compound.\')\n    rows = [{\'Compound\': compound, \'Context\': context, \'ModAvg\': \'0\', \'HeadAvg\': \'0\', \'row_key\': f\'demo:{i}\'}\n            for i, (compound, context) in enumerate(zip(compounds, contexts if contexts is not None else [\'\'] * len(compounds)))]\n    tokens, _ = tokenize_rows(rows, cfg[\'variant\'], tokenizer, cfg[\'max_length\'])\n    raw = predict(model, tokens, tokenizer, cfg[\'batch_size\'], device)\n    return [{\'raw\': pair, \'clipped\': [max(0.0, min(5.0, value)) for value in pair]} for pair in raw]\n'}
CODE_SHA256 = {'data_utils.py': 'd2d9d1fa9e93e5a71316075895279b1267fff1a02102a4135bf216fa18a17cfe', 'kaggle_en_nn.py': '8ec651f797df7bb78c1a3f864240073ddf71119c08a9dcd2aee28f53193208b5'}
CODE_DIR = OUTPUT / "code"
CODE_DIR.mkdir()
for name, source in EMBEDDED_SOURCES.items():
    if hashlib.sha256(source.encode("utf-8")).hexdigest() != CODE_SHA256[name]:
        raise ValueError("Embedded code hash mismatch")
    (CODE_DIR / name).write_text(source, encoding="utf-8")
sys.path.insert(0, str(CODE_DIR))
for name in ("kaggle_en_nn", "data_utils"):
    sys.modules.pop(name, None)
import kaggle_en_nn as pipeline
pipeline.write_json(OUTPUT / "code_hashes.json", CODE_SHA256)
print("Code loaded:", CODE_SHA256)


## 4. Tìm Dataset, kiểm tra raw TSV và replay split đã khóa

Hỗ trợ Dataset đã giải nén và Dataset còn ZIP. Không hard-code username/slug.
Cell kiểm tra hash từng file, schema, nhãn hợp lệ, row keys, coverage,
leakage context/Compound, và replay membership đúng thuật toán v2.
Không gọi random split hoặc tạo membership mới. Mọi lỗi đều dừng pipeline.


In [ ]:
bundle_root = pipeline.discover_bundle(INPUT, WORK, DATA_ROOT)
selected, integrity = pipeline.load_checked(bundle_root, SETTING)
b0 = pipeline.save_prepared(selected, integrity, OUTPUT)
pipeline.write_json(OUTPUT / "environment.json", pipeline.environment_report())
print("Bundle:", bundle_root)
print(json.dumps(integrity, indent=2, ensure_ascii=False))
print("B0 validation E:", round(b0["metrics"]["compound_macro_two_output_mae"], 6))
print("B1 inputs:", list(pipeline.model_inputs(selected["train"][0], "B1")))
print("B2 inputs:", list(pipeline.model_inputs(selected["train"][0], "B2")))
print("Prepared labels order: [ModAvg, HeadAvg]. ID/Avg/Std không nằm trong input.")


## 5. Tokenize B1/B2 và lưu dữ liệu train-ready

B1: `Compound`; B2: text pair `(Compound, Context)`.
Dùng nguyên văn nguồn; không xóa stopword, dấu câu, stemming hay sửa số nhiều.
Tokenizer `uncased` tự xử lý chữ hoa/thường theo backbone.
Không truyền token_type_ids vì DistilBERT không dùng chúng.
Chưa padding ở bước lưu; DataLoader padding động trong từng batch.
**Không tự truncate:** nếu mẫu vượt MAX_LENGTH, cell dừng để bạn xem lại cấu hình
cho cả B1/B2. Cách này tránh cắt mất cách dùng mục tiêu.


In [ ]:
from huggingface_hub import HfApi
from transformers import AutoTokenizer
resolved_revision = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION).sha
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=resolved_revision, use_fast=True)
if MAX_LENGTH > tokenizer.model_max_length:
    raise ValueError("MAX_LENGTH vượt giới hạn backbone")
tokenizer.save_pretrained(OUTPUT / "tokenizer")
model_lock = {"model_id": MODEL_ID, "requested_revision": MODEL_REVISION,
              "resolved_revision": resolved_revision, "max_length": MAX_LENGTH}
pipeline.write_json(OUTPUT / "model_lock.json", model_lock)
tokenized, token_lengths = pipeline.tokenize_all(selected, tokenizer, MAX_LENGTH, OUTPUT)
print("PIN SHA này cho các lần train sau:", resolved_revision)
print(json.dumps(token_lengths, ensure_ascii=False, indent=2))
print("Prepare hoàn tất. MODE=prepare sẽ bỏ qua optimizer ở cell kế tiếp.")


## 6. Tùy chọn: smoke hoặc train trên GPU

`prepare`: bỏ qua toàn bộ training. `smoke`: tối đa128 mẫu train, tối đa8 bước,
một epoch; validation vẫn dùng toàn bộ membership, kết quả smoke không dùng làm headline.
`train`: tối đa5 epochs, early stopping patience2; chọn best bằng MAE trung bình
theo Compound và hai output, sau clipping prediction về [0,5].
Backbone + masked mean pooling (bao gồm special tokens, loại padding) + linear2;
loss MSE, AdamW, weight decay0.01, warmup10%, linear decay, grad clipping1.0, FP32.
Hai model dùng cùng budget/config/seed và cùng pretrained revision.

Checkpoint: `best.safetensors` để inference; `last.pt` gồm model/optimizer/scheduler/RNG
để resume sau epoch đã hoàn thành. Chỉ load last.pt do chính notebook này tạo.
Interrupt giữa epoch thì phải chạy lại epoch đó; interrupt trước epoch đầu chưa có last.pt.


In [ ]:
results = []
if MODE == "prepare":
    print("Không train: đổi MODE thành smoke khi bạn sẵn sàng chạy thử GPU.")
else:
    for variant in VARIANTS:
        cfg = {"mode": MODE, "setting": SETTING, "variant": variant, "seed": SEED,
               "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE,
               "max_length": MAX_LENGTH, "epochs": EPOCHS, "patience": PATIENCE,
               "smoke_steps": SMOKE_STEPS, "model_id": MODEL_ID,
               "model_revision": resolved_revision, "code_sha256": CODE_SHA256,
               "pooling": "masked_mean_including_special_tokens", "head": "linear_two_outputs",
               "loss": "MSE", "weight_decay": 0.01, "warmup_ratio": 0.1,
               "precision": "float32", "gradient_clip_norm": 1.0}
        result = pipeline.train_variant(selected, tokenized[variant], tokenizer, cfg,
                                        OUTPUT / "runs" / variant, integrity,
                                        resume_from=RESUME_FROM.get(variant, ""))
        results.append(result)
summary = [{"model": "B0", "validation_E": b0["metrics"]["compound_macro_two_output_mae"]}]
summary.extend({"model": r["variant"], "validation_E": r["metrics"]["compound_macro_two_output_mae"],
                "mode": MODE, "seed": SEED} for r in results)
pipeline.write_json(OUTPUT / "comparison_validation.json", summary)
print(json.dumps(summary, ensure_ascii=False, indent=2))
if len(results) == 2:
    by_model = {r["variant"]: r for r in results}
    delta = (by_model["B1"]["metrics"]["compound_macro_two_output_mae"]
             - by_model["B2"]["metrics"]["compound_macro_two_output_mae"])
    print("Delta E(B1)-E(B2):", delta, "| dương = B2 ít sai hơn; một seed chưa đủ kết luận ổn định")


## 7. Kiểm tra load lại checkpoint và inference

Nạp lại đầy đủ custom regression head, config và tokenizer.
So ba prediction validation đầu với prediction đã lưu; không đụng final.
Nếu chỉ prepare thì cell bỏ qua. Web sau này có thể dùng `predict_texts` cùng checkpoint này.


In [ ]:
if results:
    import csv
    for result in results:
        run_dir = OUTPUT / "runs" / result["variant"]
        probe = selected["validation"][:3]
        fresh = pipeline.predict_texts(run_dir, [r["Compound"] for r in probe],
                                      [r["Context"] for r in probe], device="cuda:0")
        with (run_dir / "validation_predictions.tsv").open(encoding="utf-8", newline="") as handle:
            saved = list(csv.DictReader(handle, delimiter="\t"))[:3]
        for actual, expected in zip(fresh, saved):
            for value, column in zip(actual["raw"], ["raw_ModAvg", "raw_HeadAvg"]):
                if abs(value - float(expected[column])) > 1e-4:
                    raise ValueError("Reload prediction mismatch")
        print(result["variant"], "reload/inference PASS")
    pipeline.write_json(OUTPUT / "reload_check.json", {"passed": True, "final_evaluated": False})
else:
    print("Prepare only: chưa có checkpoint để reload.")


## 8. Đóng gói kết quả để tải về

ZIP chỉ chứa thư mục output của lần chạy này. Không gom toàn bộ /kaggle/working,
không kèm credentials, input Dataset hoặc cache tải model.
Prepared data và prediction gold là dữ liệu dẫn xuất: tiếp tục giữ private.
Checkpoint có thể lớn vì last.pt chứa optimizer. Tải về trước khi hết session;
lưu một private Notebook Version để giữ output lâu dài.


In [ ]:
import shutil
from IPython.display import FileLink, display
archive = Path(shutil.make_archive(str(WORK / OUTPUT.name), "zip", root_dir=OUTPUT.parent, base_dir=OUTPUT.name))
print("Output:", OUTPUT)
print("ZIP:", archive, "| bytes:", archive.stat().st_size)
print("SHA256:", pipeline.digest(archive))
# Link tương đối để Jupyter/Kaggle phục vụ file từ working directory.
os.chdir(WORK)
display(FileLink(archive.name))


## Cách đọc kết quả và lượt chạy tiếp theo

B0 tham chiếu (validation E): B≈0.670054; A≈0.745341. E càng thấp càng tốt,
so B0/B1/B2 **trong cùng setting**, không dùng chênh lệch A/B như một hiệu ứng nhân quả.
Đây là metric nội bộ, chưa xác minh là metric leaderboard.

Sau prepare → smoke → train B1/B2 seed42 trên B → đọc lỗi validation.
Sau đó chạy A và lặp seed13,2027 cho các so sánh quan trọng.
Tuning chỉ trên validation: grid1e-5,2e-5,3e-5, tối đa3 trial/model/setting,
lưu mọi trial và áp dụng budget công bằng. Không chỉ báo best seed.
Chốt lựa chọn chung A/B trước khi mở bất kỳ final nào.

Tài liệu chính thức được dùng để soạn hướng dẫn (đối chiếu17/10/2026):
[Kaggle Datasets](https://www.kaggle.com/docs/datasets),
[Kaggle Notebooks](https://www.kaggle.com/docs/notebooks),
[HF padding/truncation4.57.1](https://huggingface.co/docs/transformers/v4.57.1/en/pad_truncation),
[Auto Classes4.57.1](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/auto).
